In [ ]:
import numpy as np 
import pandas as pd 
import os
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from tqdm import tqdm

TRAIN_CSV = "/kaggle/input/campus-images/labels_train_updated.csv"
VAL_CSV   = "/kaggle/input/campus-images/labels_val_updated.csv"
TRAIN_IMG_DIR = "/kaggle/input/campus-images/images_train/images_train"
VAL_IMG_DIR   = "/kaggle/input/campus-images/images_val/images_val"
TEST_IMG_DIR  = "/kaggle/input/testdata/images_test"



In [ ]:
class LatLongDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, scaling_params=None,is_train=True):
        dataframe = pd.read_csv(csv_file)

        if is_train:
            
            lat_low, lat_high = dataframe['latitude'].quantile(0.02), dataframe['latitude'].quantile(0.98)
            lon_low, lon_high = dataframe['longitude'].quantile(0.02), dataframe['longitude'].quantile(0.98)
            dataframe = dataframe[
                dataframe['latitude'].between(lat_low, lat_high) &
                dataframe['longitude'].between(lon_low, lon_high)
            ].reset_index(drop=True)

        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

        if scaling_params is None:
            self.lat_mean = dataframe['latitude'].mean()
            self.lat_std  = dataframe['latitude'].std()
            self.lon_mean = dataframe['longitude'].mean()
            self.lon_std  = dataframe['longitude'].std()
        else:
            self.lat_mean, self.lat_std, self.lon_mean, self.lon_std = scaling_params

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        if self.transform:
            img = self.transform(img)

        lat_norm = (row['latitude'] - self.lat_mean) / self.lat_std
        lon_norm = (row['longitude'] - self.lon_mean) / self.lon_std
        norm_target = torch.tensor([lat_norm, lon_norm], dtype=torch.float32)
        orig_target = torch.tensor([row['latitude'], row['longitude']], dtype=torch.float32)

        return img, norm_target, orig_target
    
class LatLongPredictionConvNeXt(nn.Module):
    def __init__(self, model_name='convnext_base', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        
        _, C, H, W = self.backbone(torch.zeros(1,3,224,224))[ -1 ].shape
        in_features = C

        # define pooling + MLP head
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  
            nn.Flatten(),             
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        feats = self.backbone(x)[-1]  # last feature map
        return self.head(feats)

In [ ]:

# Transforms
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Compute scaling params on filtered train set
train_dataframe = pd.read_csv(TRAIN_CSV)
lat_low, lat_high = train_dataframe['latitude'].quantile(0.02), train_dataframe['latitude'].quantile(0.98)
lon_low, lon_high = train_dataframe['longitude'].quantile(0.02), train_dataframe['longitude'].quantile(0.98)
filt = train_dataframe[
    train_dataframe['latitude'].between(lat_low, lat_high) &
    train_dataframe['longitude'].between(lon_low, lon_high)
]
scaling_params = (
    filt['latitude'].mean(), filt['latitude'].std(),
    filt['longitude'].mean(), filt['longitude'].std()
)


batch_size = 32
train_dataset = LatLongDataset(TRAIN_CSV, TRAIN_IMG_DIR, transform_train, scaling_params)
val_dataset   = LatLongDataset(VAL_CSV,   VAL_IMG_DIR,   transform_val,   scaling_params)

print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size, shuffle=False, num_workers=2)

print(f"Data Loading done...")

In [ ]:
# Initialize model
net = LatLongPredictionConvNeXt('convnext_base', pretrained=True)

# Set computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

# Enable multi-GPU if available
if torch.cuda.device_count() > 1:
    print(f"Multiple GPUs detected: {torch.cuda.device_count()}")
    net = nn.DataParallel(net)

net.to(device)

# Define loss function and optimizer
loss_fn = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=1e-4)

# Training parameters
epochs = 20
lowest_val_loss = float('inf')

# Training loop
for ep in range(epochs):
    net.train()
    total_train_loss_norm = 0.0
    total_train_loss_orig = 0.0

    for batch in tqdm(train_loader, desc=f"Training Epoch {ep+1}", leave=False):
        images, norm_labels, orig_labels = batch
        images, norm_labels, orig_labels = images.to(device), norm_labels.to(device), orig_labels.to(device)

        optimizer.zero_grad()
        norm_preds = net(images)
        loss = loss_fn(norm_preds, norm_labels)
        loss.backward()
        optimizer.step()

        total_train_loss_norm += loss.item() * images.size(0)

        # Denormalize predictions
        denorm_preds = norm_preds.clone()
        denorm_preds[:, 0] = denorm_preds[:, 0] * scaling_params[1] + scaling_params[0]
        denorm_preds[:, 1] = denorm_preds[:, 1] * scaling_params[3] + scaling_params[2]
        total_train_loss_orig += ((denorm_preds - orig_labels) ** 2).sum().item()

    avg_train_loss_norm = total_train_loss_norm / len(train_dataset)
    avg_train_loss_orig = total_train_loss_orig / len(train_dataset)

    # Validation
    net.eval()
    total_val_loss_norm = 0.0
    total_val_loss_orig = 0.0

    with torch.no_grad():
        for val_batch in tqdm(val_loader, desc=f"Validation Epoch {ep+1}", leave=False):
            val_images, val_norms, val_origs = val_batch
            val_images, val_norms, val_origs = val_images.to(device), val_norms.to(device), val_origs.to(device)

            val_preds = net(val_images)
            total_val_loss_norm += loss_fn(val_preds, val_norms).item() * val_images.size(0)

            val_preds_denorm = val_preds.clone()
            val_preds_denorm[:, 0] = val_preds_denorm[:, 0] * scaling_params[1] + scaling_params[0]
            val_preds_denorm[:, 1] = val_preds_denorm[:, 1] * scaling_params[3] + scaling_params[2]
            total_val_loss_orig += ((val_preds_denorm - val_origs) ** 2).sum().item()

    avg_val_loss_norm = total_val_loss_norm / len(val_dataset)
    avg_val_loss_orig = total_val_loss_orig / len(val_dataset)

    print(
        f"Epoch [{ep+1}/{epochs}] -> "
        f"Train Loss (Norm): {avg_train_loss_norm:.4f}, Train Loss (Orig): {avg_train_loss_orig:.4f} | "
        f"Val Loss (Norm): {avg_val_loss_norm:.4f}, Val Loss (Orig): {avg_val_loss_orig:.4f}"
    )

    # Save best model based on normalized validation loss
    if avg_val_loss_norm < lowest_val_loss:
        lowest_val_loss = avg_val_loss_norm
        model_state = net.module.state_dict() if isinstance(net, nn.DataParallel) else net.state_dict()
        torch.save(model_state, "best_model.pth")
        print(f"Model saved with Val Norm Loss: {lowest_val_loss:.4f}")


In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import timm
from tqdm import tqdm
import uuid

# Custom Dataset for Test (no CSV)
class TestLatLongDataset(Dataset):
    def __init__(self, img_dir, transform=None, start_idx=369):
        self.img_dir = img_dir
        self.transform = transform
        # Get all image files (jpg, png, etc.) in the directory
        self.filenames = [
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        # Ensure we have exactly 369 images
        assert len(self.filenames) == 369, f"Expected 369 images, found {len(self.filenames)}"
        self.filenames.sort()  # Sort for consistency
        self.ids = list(range(start_idx, start_idx + len(self.filenames)))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.filenames[idx])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.ids[idx]

# Validation Dataset (same as before, but only need image and ID)
class ValLatLongDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, idx  # Return index as ID

# Model Definition (same as training)
class LatLongPredictionConvNeXt(nn.Module):
    def __init__(self, model_name='convnext_base', pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        _, C, H, W = self.backbone(torch.zeros(1, 3, 224, 224))[-1].shape
        in_features = C
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        feats = self.backbone(x)[-1]
        return self.head(feats)

# Setup
MODEL_PATH = "/kaggle/working/latlong_convnext_best.pth"
OUTPUT_CSV = "predictions.csv"
VAL_CSV   = "/kaggle/input/campus-images/labels_val_updated.csv"
VAL_IMG_DIR   = "/kaggle/input/campus-images/images_val/images_val"
TEST_IMG_DIR  = "/kaggle/input/testdata/images_test"

# Transformations (same as validation in training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load scaling parameters (recompute from filtered train set, as in training)
train_df = pd.read_csv("/kaggle/input/campus-images/labels_train_updated.csv")  # Update with actual path
lat_lo, lat_hi = train_df['latitude'].quantile(0.02), train_df['latitude'].quantile(0.98)
lon_lo, lon_hi = train_df['longitude'].quantile(0.02), train_df['longitude'].quantile(0.98)
filt = train_df[
    train_df['latitude'].between(lat_lo, lat_hi) &
    train_df['longitude'].between(lon_lo, lon_hi)
]
scaling_params = (
    filt['latitude'].mean(), filt['latitude'].std(),
    filt['longitude'].mean(), filt['longitude'].std()
)

# Initialize datasets and dataloaders
batch_size = 32
val_ds = ValLatLongDataset(VAL_CSV, VAL_IMG_DIR, transform)
test_ds = TestLatLongDataset(TEST_IMG_DIR, transform, start_idx=369)
val_loader = DataLoader(val_ds, batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size, shuffle=False, num_workers=2)

# Initialize model and load weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LatLongPredictionConvNeXt('convnext_base', pretrained=False)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# Generate predictions
predictions = []
with torch.no_grad():
    # Validation predictions
    for imgs, ids in tqdm(val_loader, desc="Predicting Validation"):
        imgs = imgs.to(device)
        preds_norm = model(imgs)
        # Denormalize predictions
        preds = preds_norm.clone()
        preds[:, 0] = preds[:, 0] * scaling_params[1] + scaling_params[0]  # Latitude
        preds[:, 1] = preds[:, 1] * scaling_params[3] + scaling_params[2]  # Longitude
        for id_val, pred in zip(ids, preds.cpu().numpy()):
            predictions.append([int(id_val), pred[0], pred[1]])

    # Test predictions
    for imgs, ids in tqdm(test_loader, desc="Predicting Test"):
        imgs = imgs.to(device)
        preds_norm = model(imgs)
        # Denormalize predictions
        preds = preds_norm.clone()
        preds[:, 0] = preds[:, 0] * scaling_params[1] + scaling_params[0]  # Latitude
        preds[:, 1] = preds[:, 1] * scaling_params[3] + scaling_params[2]  # Longitude
        for id_val, pred in zip(ids, preds.cpu().numpy()):
            predictions.append([int(id_val), pred[0], pred[1]])

# Save predictions to CSV
predictions_df = pd.DataFrame(predictions, columns=['id', 'Latitude', 'Longitude'])
predictions_df.to_csv(OUTPUT_CSV, index=False)
print(f"Predictions saved to {OUTPUT_CSV}")